<a href="https://colab.research.google.com/github/nishakamble178/SwarSetu_DL/blob/main/SwarSetu%5BDL%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Connect and allow access from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Import Path of the datasets
dataset_path = "/content/drive/MyDrive/SwarSetu_Dataset"

In [ ]:
#show the content of the Drive
import os
print(os.listdir('/content/drive/MyDrive/SwarSetu_Dataset'))

['Rock_Paper', 'Numbers', 'Alphabets', 'Gestures']


In [ ]:
#Scan the data into the batches
import os

dataset_path = "/content/drive/MyDrive/SwarSetu_Dataset"

image_paths = []
labels = []

ignore = ['train','test','validation','valid','data']

print("⏳ Scanning dataset...\n")

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.jpg','.jpeg','.png')):

            # Clean label
            path_parts = root.split(os.sep)
            clean_parts = [p for p in path_parts if p.lower() not in ignore]

            label = clean_parts[-1]

            # Clean gesture labels (03_fist → fist)
            if "_" in label:
                label = label.split("_")[-1]

            image_paths.append(os.path.join(root, file))
            labels.append(label)

print("✅ Scanning finished!\n")

# 🎯 UNIQUE CLASSES
unique_classes = sorted(set(labels))

print("📂 Classes in the Dataset :\n")
for cls in unique_classes:
    print("➡️", cls)

print("\n📊 Total images:", len(image_paths))
print("📊 Total classes:", len(unique_classes))

⏳ Scanning dataset...

✅ Scanning finished!

📂 Classes (line by line):

➡️ 
➡️ 0
➡️ 1
➡️ 2
➡️ 3
➡️ 4
➡️ 5
➡️ 6
➡️ 7
➡️ 8
➡️ 9
➡️ A
➡️ B
➡️ C
➡️ D
➡️ E
➡️ F
➡️ G
➡️ H
➡️ I
➡️ J
➡️ K
➡️ L
➡️ M
➡️ N
➡️ O
➡️ P
➡️ Q
➡️ R
➡️ S
➡️ T
➡️ U
➡️ V
➡️ W
➡️ X
➡️ Y
➡️ Z
➡️ fist
➡️ five
➡️ none
➡️ okay
➡️ paper
➡️ peace
➡️ rad
➡️ rock
➡️ scissors
➡️ straight
➡️ thumbs
➡️ unknown

📊 Total images: 107426
📊 Total classes: 49


In [ ]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

L=LabelEncoder()
L_encoded=L.fit_transform(labels)
L_Categorical=to_categorical(L_encoded)
print("✅Labels Encoded")

✅Labels Encoded


In [ ]:
#Create a generator
import numpy as np
import cv2
import random

def Data_generator(image_paths,labels,batch_size=32):

  while True:
    combine=list(zip(image_paths,labels))
    random.shuffle(combine)
    image_paths,labels=zip(*combine)

    for i in range(0,len(image_paths),batch_size):
      batch_paths=image_paths[i:i+batch_size]
      batch_labels=labels[i:i+batch_size]
      batch_images=[]

      for path in batch_paths:
        image=cv2.imread(path)
        if image is not None:
          image=cv2.resize(image,(64,64))
          image=image/255.0
          batch_images.append(image)
      yield np.array(batch_images),np.array(batch_labels)

In [ ]:
gen = Data_generator(image_paths, L_Categorical, 32)

X, y = next(gen)

print("Images shape:", X.shape)
print("Labels shape:", y.shape)

Images shape: (32, 64, 64, 3)
Labels shape: (32, 49)


It Will Give you the How Generator will be work

it takes 32 images and each images have 64*64 pixel

Each pixel have 3 colors Red,Green and Blue.

In [ ]:
#Here We have to Build the actual model of CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

num_classes = L_Categorical.shape[1]

model = Sequential()

model.add(Input(shape=(64,64,3)))  # ✅ correct modern way
model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

#Convert the data into the 1D
model.add(Flatten())

#Learning phase
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

#Output layer
model.add(Dense(num_classes, activation='softmax'))

#Here the compilation
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 49)             │         6,321 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 689,521 (2.63 MB)

 Trainable params: 689,521 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

* Conv2D[32img] :simply Look the Height[62],Width[62] and channels[32].
* MAxpooling : Make img smaller only keep the imp parts
* Conv2D[64img] : Find the better shapes and curves.
*  Maxpooling : reduce again to go faster
*  Conv2D[128]: complex img
* Flatten : convert Img into the Numbers
* Dense : Think and learn the patterns
* Dropout :Learn properly













In [ ]:
#To check the accuracy for the Confirm that model is train or not.
batch_size = 64

train_gen = Data_generator(image_paths, L_Categorical, batch_size)

steps_per_epoch = 50   # keep small for speed
epochs = 1              # start small

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=epochs
)

50/50 ━━━━━━━━━━━━━━━━━━━━ 170s 3s/step - accuracy: 0.7588 - loss: 0.7612


The model Got 76% accuracy only 50 steps and only 1 epoch.
That means the model is learning correctly
Dataset & Pipelines is corret

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch=200,
    epochs=2               #increase
)

Epoch 1/2
200/200 ━━━━━━━━━━━━━━━━━━━━ 718s 4s/step - accuracy: 0.7920 - loss: 0.6419
Epoch 2/2
200/200 ━━━━━━━━━━━━━━━━━━━━ 770s 4s/step - accuracy: 0.8270 - loss: 0.5248


Epoch 1 → accuracy: 0.79 (79%)

Epoch 2 → accuracy: 0.82 (82%)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    image_paths, L_Categorical, test_size=0.2, random_state=42
)

train_gen = Data_generator(X_train, y_train, 64)
val_gen = Data_generator(X_val, y_val, 64)

Data is split into training (80%) and validation (20%)

Data generator creates batches of size 64

Output is not printed directly, but used during model training



In [ ]:
model.fit(
    train_gen,
    steps_per_epoch=200,
    validation_data=val_gen,
    validation_steps=50,
    epochs=4
)

Epoch 1/4
200/200 ━━━━━━━━━━━━━━━━━━━━ 424s 2s/step - accuracy: 0.8965 - loss: 0.3138 - val_accuracy: 0.9619 - val_loss: 0.1316
Epoch 2/4
200/200 ━━━━━━━━━━━━━━━━━━━━ 386s 2s/step - accuracy: 0.9049 - loss: 0.2893 - val_accuracy: 0.9575 - val_loss: 0.1486
Epoch 3/4
200/200 ━━━━━━━━━━━━━━━━━━━━ 415s 2s/step - accuracy: 0.9054 - loss: 0.2813 - val_accuracy: 0.9663 - val_loss: 0.1207
Epoch 4/4
200/200 ━━━━━━━━━━━━━━━━━━━━ 390s 2s/step - accuracy: 0.9104 - loss: 0.2700 - val_accuracy: 0.9616 - val_loss: 0.1252


Training accuracy: ~91%

Validation accuracy: ~96%



In [ ]:
from google.colab import files
files.upload()

 Train Model is saved properly !!!

 Keras is a High-level deep learning library it create AI models without writing very complex math code.

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("final_model.keras")
print("✅ Model loaded successfully")

✅ Model loaded successfully


In [ ]:
import numpy as np

dummy_input = np.random.random((1, 64, 64, 3))
prediction = model.predict(dummy_input)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step
[[8.53694075e-07 1.25331819e-04 6.89785975e-06 5.99985011e-04
  3.28665599e-04 4.00109966e-05 3.59200378e-04 2.70142627e-04
  6.52783228e-07 1.18757441e-06 1.45397452e-03 2.83369736e-04
  1.84347107e-06 1.60494503e-02 2.45834468e-04 9.71067711e-05
  5.35739586e-04 7.25331347e-06 2.67475502e-06 5.49423276e-03
  1.98064707e-02 6.54219502e-06 9.18518528e-02 1.19954029e-05
  7.34407786e-06 8.24728049e-05 3.37896527e-05 1.54627385e-04
  3.73006586e-08 2.20144975e-05 3.48185276e-05 7.40906253e-05
  8.70312564e-03 1.04935592e-04 1.11503046e-04 2.45181582e-04
  1.98015757e-03 6.08123664e-04 5.53563237e-03 1.99123335e-07
  8.57941508e-02 2.34985650e-01 2.78209336e-04 2.34668638e-04
  1.40620964e-02 1.37256399e-01 1.83048951e-06 9.09494353e-04
  3.71198118e-01]]
